# 27 · Phase H1c — Physical-Shift Decomposition

This notebook decomposes the nonredundant H1b physical-state shift by year, class, active region, physical family, source boundary and frozen statistical agreement signals.

**No routing thresholds are selected or changed.** Cycle-25 and 2026 remain retrospective/post-hoc evidence.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

ROOT=Path("/home/abmoses2000")
H1B=ROOT/"graybox_aia72_physical_state_h1b_nonredundant_v1_20261006"
FULL=H1B/"physical_state_full_113433.csv.gz"
MM=H1B/"physical_multimodal_case_table.csv.gz"
LEDGER=ROOT/"sharp72_predictions_frozen_20261002.csv.gz"
MASTER=ROOT/"graybox_aia72_master_predictions_v2_20261005"/"graybox_aia72_master_predictions.csv.gz"
OUT=ROOT/"graybox_aia72_physical_shift_h1c_v1_20261006"

for p in [FULL,MM,LEDGER,MASTER]:
    assert p.is_file(), f"Missing required input: {p}"
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(4*1024*1024),b""): h.update(b)
    return h.hexdigest()

full=pd.read_csv(FULL,low_memory=False)
mm=pd.read_csv(MM,low_memory=False)
ledger=pd.read_csv(LEDGER,low_memory=False)
master=pd.read_csv(MASTER,low_memory=False)

for df in [full,mm,ledger,master]:
    assert "forecast_case_id" in df.columns
    df["forecast_case_id"]=df["forecast_case_id"].astype(str)

full["issue_dt"]=pd.to_datetime(full["issue_utc"],utc=True,errors="raise")
full["year"]=full["issue_dt"].dt.year.astype(int)
known=full[full["label_known"].astype(bool) & full["label"].isin([0,1])].copy()

MPSV=[c for c in full.columns if "__" in c and not c.startswith("physical_")]
assert len(MPSV)==15, MPSV

print("H1b full:",full.shape)
print("Known-label cases:",known.shape)
print("MPSV axes:",MPSV)


In [ ]:
# H1c-A: year-by-year applicability
year_rows=[]
for yr,g in known.groupby("year",sort=True):
    y=g["label"].to_numpy(int)
    d=g["physical_md2"].to_numpy(float)
    year_rows.append({
      "year":int(yr),
      "cases":int(len(g)),
      "active_regions":int(g["region_component_id"].nunique()),
      "positives":int(y.sum()),
      "prevalence":float(y.mean()),
      "median_md2":float(np.median(d)),
      "q25_md2":float(np.quantile(d,0.25)),
      "q75_md2":float(np.quantile(d,0.75)),
      "ood_rate_q99":float(g["physical_ood_q99"].mean()),
    })
year=pd.DataFrame(year_rows)
year.to_csv(OUT/"physical_applicability_by_year_detailed.csv",index=False)

# H1c-B: class-conditional applicability by regime and year
class_rows=[]
for keys,g in known.groupby(["physical_regime","label"],sort=False):
    reg,label=keys
    class_rows.append({
      "grouping":"regime","group":reg,"label":int(label),
      "cases":int(len(g)),
      "median_md2":float(g["physical_md2"].median()),
      "q25_md2":float(g["physical_md2"].quantile(.25)),
      "q75_md2":float(g["physical_md2"].quantile(.75)),
      "ood_rate_q99":float(g["physical_ood_q99"].mean()),
      "median_physical_probability":float(g["physical_probability"].median()),
    })
for keys,g in known.groupby(["year","label"],sort=True):
    yr,label=keys
    class_rows.append({
      "grouping":"year","group":str(int(yr)),"label":int(label),
      "cases":int(len(g)),
      "median_md2":float(g["physical_md2"].median()),
      "q25_md2":float(g["physical_md2"].quantile(.25)),
      "q75_md2":float(g["physical_md2"].quantile(.75)),
      "ood_rate_q99":float(g["physical_ood_q99"].mean()),
      "median_physical_probability":float(g["physical_probability"].median()),
    })
class_summary=pd.DataFrame(class_rows)
class_summary.to_csv(OUT/"physical_applicability_class_conditional.csv",index=False)

print("===== YEAR-BY-YEAR =====")
print(year.to_string(index=False))
print("\n===== CLASS-CONDITIONAL BY REGIME =====")
print(class_summary[class_summary.grouping.eq("regime")].to_string(index=False))


In [ ]:
# H1c-C: active-region-level sensitivity
# Aggregate repeated 96-min windows before summarising regimes.
ar=known.groupby(["physical_regime","year","region_component_id"],dropna=False).agg(
    windows=("forecast_case_id","size"),
    any_positive=("label","max"),
    positive_windows=("label","sum"),
    median_md2=("physical_md2","median"),
    max_md2=("physical_md2","max"),
    ood_fraction=("physical_ood_q99","mean"),
).reset_index()

ar.to_csv(OUT/"physical_applicability_by_active_region.csv",index=False)

ar_reg=[]
for reg,g in ar.groupby("physical_regime",sort=False):
    ar_reg.append({
      "regime":reg,
      "active_regions":int(len(g)),
      "positive_active_regions":int(g["any_positive"].sum()),
      "median_ar_median_md2":float(g["median_md2"].median()),
      "median_ar_ood_fraction":float(g["ood_fraction"].median()),
      "fraction_ar_any_ood":float((g["ood_fraction"]>0).mean()),
      "median_windows_per_ar":float(g["windows"].median()),
    })
ar_reg=pd.DataFrame(ar_reg)
ar_reg.to_csv(OUT/"physical_applicability_ar_regime_summary.csv",index=False)

print("===== AR-LEVEL REGIME SUMMARY =====")
print(ar_reg.to_string(index=False))


In [ ]:
# H1c-D: physical-family/axis shift decomposition
train=full[full["role"].eq("train")].copy()
axis_rows=[]
for c in MPSV:
    tr=train[c].to_numpy(float)
    tr_med=float(np.median(tr))
    tr_iqr=float(np.quantile(tr,.75)-np.quantile(tr,.25))
    if tr_iqr<=1e-12: tr_iqr=np.nan
    for reg,g in full[full["physical_regime"].ne("unknown_label_excluded")].groupby("physical_regime",sort=False):
        med=float(g[c].median())
        axis_rows.append({
          "axis":c,"grouping":"regime","group":reg,"cases":int(len(g)),
          "median":med,"iqr":float(g[c].quantile(.75)-g[c].quantile(.25)),
          "training_median":tr_med,"training_iqr":tr_iqr,
          "standardized_median_shift":float((med-tr_med)/tr_iqr) if np.isfinite(tr_iqr) else np.nan,
        })
    for yr,g in full.groupby("year",sort=True):
        med=float(g[c].median())
        axis_rows.append({
          "axis":c,"grouping":"year","group":str(int(yr)),"cases":int(len(g)),
          "median":med,"iqr":float(g[c].quantile(.75)-g[c].quantile(.25)),
          "training_median":tr_med,"training_iqr":tr_iqr,
          "standardized_median_shift":float((med-tr_med)/tr_iqr) if np.isfinite(tr_iqr) else np.nan,
        })

axis=pd.DataFrame(axis_rows)
axis.to_csv(OUT/"physical_axis_shift_decomposition.csv",index=False)

rank=(axis[(axis.grouping=="regime")&(axis.group=="supplementary_2026_posthoc")]
      .assign(abs_shift=lambda d:d.standardized_median_shift.abs())
      .sort_values("abs_shift",ascending=False))
rank.to_csv(OUT/"physical_axis_2026_shift_ranking.csv",index=False)

print("===== 2026 AXIS SHIFT RANKING =====")
print(rank[["axis","median","standardized_median_shift"]].to_string(index=False))


In [ ]:
# H1c-E: source/time-boundary audit.
# Monthly summaries reveal whether the shift is gradual or abrupt; no window is selected by outcome.
monthly=known[known["year"].isin([2024,2025,2026])].copy()
monthly["month"]=monthly["issue_dt"].dt.to_period("M").astype(str)
month_rows=[]
for month,g in monthly.groupby("month",sort=True):
    month_rows.append({
      "month":month,
      "cases":int(len(g)),
      "active_regions":int(g["region_component_id"].nunique()),
      "positives":int(g["label"].sum()),
      "prevalence":float(g["label"].mean()),
      "median_md2":float(g["physical_md2"].median()),
      "ood_rate_q99":float(g["physical_ood_q99"].mean()),
    })
month=pd.DataFrame(month_rows)
month.to_csv(OUT/"physical_applicability_monthly_2024_2026.csv",index=False)

print("===== MONTHLY 2024-2026 =====")
print(month.to_string(index=False))


In [ ]:
# H1c-F: contrast physical applicability with frozen statistical agreement.
# Rebuild exact multimodal agreement signals from the two frozen probability sources.
master["forecast_case_id"]=master["forecast_case_id"].astype(str)
ledger["forecast_case_id"]=ledger["forecast_case_id"].astype(str)

m=master.merge(
    ledger[["forecast_case_id","probability_seed_17","probability_seed_29","probability_seed_43"]],
    on="forecast_case_id",how="inner",suffixes=("_aia","_sharp"),validate="one_to_one"
)
m=m.merge(
    full[["forecast_case_id","physical_md2","physical_regime"]],
    on="forecast_case_id",how="inner",validate="one_to_one"
)
assert len(m)==71010

SEEDS=[17,29,43]
aia_cols=[f"probability_seed_{z}_aia" for z in SEEDS]
sharp_cols=[f"probability_seed_{z}_sharp" for z in SEEDS]
assert all(c in m.columns for c in aia_cols+sharp_cols)

aia_seed=m[aia_cols].to_numpy(float)
sharp_seed=m[sharp_cols].to_numpy(float)
fusion_seed=.5*aia_seed+.5*sharp_seed
m["fusion_seed_std"]=np.nanstd(fusion_seed,axis=1,ddof=1)

m["aia_probability"]=pd.to_numeric(m["probability"],errors="coerce")
m["sharp_probability"]=pd.to_numeric(m["probability_gru_mean"],errors="coerce")
m["fusion_probability"]=.5*m["aia_probability"]+.5*m["sharp_probability"]
m["sharp_aia_probability_gap"]=np.abs(m["sharp_probability"]-m["aia_probability"])
q=np.clip(m["fusion_probability"].to_numpy(float),1e-12,1-1e-12)
m["fusion_entropy"]=-(q*np.log(q)+(1-q)*np.log(1-q))

contrast=[]
for reg,g in m.groupby("physical_regime",sort=False):
    contrast.append({
      "regime":reg,"cases":int(len(g)),
      "median_physical_md2":float(g["physical_md2"].median()),
      "median_fusion_seed_std":float(g["fusion_seed_std"].median()),
      "median_fusion_entropy":float(g["fusion_entropy"].median()),
      "median_sharp_aia_gap":float(g["sharp_aia_probability_gap"].median()),
      "spearman_md2_seedstd":float(g["physical_md2"].corr(g["fusion_seed_std"],method="spearman")),
      "spearman_md2_entropy":float(g["physical_md2"].corr(g["fusion_entropy"],method="spearman")),
      "spearman_md2_gap":float(g["physical_md2"].corr(g["sharp_aia_probability_gap"],method="spearman")),
    })
contrast=pd.DataFrame(contrast)
contrast.to_csv(OUT/"physical_vs_statistical_agreement.csv",index=False)

print("===== PHYSICAL VS STATISTICAL AGREEMENT =====")
print(contrast.to_string(index=False))


In [ ]:
# H1c is diagnostic only; write a receipt without a policy conclusion.
receipt={
 "status":"PHASE_H1C_SHIFT_DECOMPOSITION_COMPLETE",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "policy_changed":False,
 "later_regime_evidence":"posthoc_already_seen_in_A_to_G",
 "questions":[
   "year-by-year physical applicability",
   "class-conditional physical applicability",
   "active-region-level sensitivity",
   "physical-axis shift decomposition",
   "2024-2026 source/time-boundary audit",
   "physical versus statistical agreement contrast",
 ],
 "source_sha256":{
   "h1b_full":sha256(FULL),
   "h1b_multimodal":sha256(MM),
   "sharp_ledger":sha256(LEDGER),
   "multimodal_master":sha256(MASTER),
 },
 "policy_action":"NONE",
 "allowed_next_step":"Review H1c before any H2 physical trust-gate protocol.",
}
(OUT/"h1c_receipt.json").write_text(json.dumps(receipt,indent=2)+"\n")
print(json.dumps(receipt,indent=2))
print("Outputs:",OUT)
